# Hybrid Quantum-Classical Image Classifier
## Data Preprocessing & Feature Extraction Pipeline

This notebook demonstrates the complete classical feature extraction pipeline that prepares MNIST image data for quantum processing.

**Pipeline Overview:**
1. Load and explore MNIST dataset
2. Visualize sample images and distributions
3. Train classical CNN feature extractor
4. Extract and visualize learned features
5. Reduce features to quantum-compatible dimensions (4×4 → 16 features)
6. Compare with PCA-based dimensionality reduction

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import sys
import os

sys.path.insert(0, '..')

from src.feature_extractor import ClassicalCNNFeatureExtractor, PCAFeatureExtractor
from src.utils import plot_features

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries loaded successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load and Explore MNIST Dataset

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load training and test datasets
train_dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='../data', train=False, download=True, transform=transform)

# Filter for binary classification (0 vs 1)
train_indices = (train_dataset.targets == 0) | (train_dataset.targets == 1)
test_indices = (test_dataset.targets == 0) | (test_dataset.targets == 1)

train_dataset.data = train_dataset.data[train_indices]
train_dataset.targets = train_dataset.targets[train_indices]
test_dataset.data = test_dataset.data[test_indices]
test_dataset.targets = test_dataset.targets[test_indices]

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Class distribution (train):")
for cls in [0, 1]:
    count = (train_dataset.targets == cls).sum().item()
    print(f"  Class {cls}: {count}")

## 2. Visualize Sample Images

In [ ]:
# Display sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample MNIST Images (Classes 0 and 1)', fontsize=14, fontweight='bold')

# Get samples from each class
class_0_indices = np.where(train_dataset.targets == 0)[0][:5]
class_1_indices = np.where(train_dataset.targets == 1)[0][:5]

for i, idx in enumerate(class_0_indices):
    image = train_dataset.data[idx].numpy()
    axes[0, i].imshow(image, cmap='gray')
    axes[0, i].set_title(f'Class 0')
    axes[0, i].axis('off')

for i, idx in enumerate(class_1_indices):
    image = train_dataset.data[idx].numpy()
    axes[1, i].imshow(image, cmap='gray')
    axes[1, i].set_title(f'Class 1')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("✓ Sample images displayed")

## 3. Train Classical CNN Feature Extractor

In [ ]:
# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Create feature extractor
extractor = ClassicalCNNFeatureExtractor(output_size=16, device=device)
print(f"\n✓ Feature extractor created")
print(f"  Output features: 16 (for 4×4 quantum encoding)")
print(f"  Architecture:")
print(extractor)

In [ ]:
# Define loss and optimizer for feature extractor training
# We'll train it as a simple classifier first
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(extractor.parameters(), lr=1e-3)

# Note: For production, this should be pre-trained or trained end-to-end with the quantum classifier
# Here we just demonstrate feature extraction

print("✓ Optimizer and loss function configured")

## 4. Extract Features from Batch

In [ ]:
# Extract features from a sample batch
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    sample_features = extractor(sample_images)

print(f"Sample batch statistics:")
print(f"  Input shape: {sample_images.shape} (batch_size, channels, height, width)")
print(f"  Extracted features shape: {sample_features.shape} (batch_size, n_features)")
print(f"  Feature statistics:")
print(f"    Min: {sample_features.min().item():.4f}")
print(f"    Max: {sample_features.max().item():.4f}")
print(f"    Mean: {sample_features.mean().item():.4f}")
print(f"    Std: {sample_features.std().item():.4f}")

## 5. Visualize Feature Distributions

In [ ]:
# Extract features for all training data
all_features = []
all_labels = []

extractor.eval()
with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        features = extractor(images)
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())

all_features = np.concatenate(all_features, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

print(f"Extracted features for {len(all_features)} samples")
print(f"Feature matrix shape: {all_features.shape}")

In [ ]:
# Visualize features in 2D using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
features_2d = pca.fit_transform(all_features)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], 
                       c=all_labels, cmap='viridis', alpha=0.6, s=50)
plt.colorbar(scatter, label='Class')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('CNN-Extracted Features (2D PCA Projection)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Explained variance by first 2 PCs: {pca.explained_variance_ratio_.sum():.2%}")

## 6. Compare with PCA-based Feature Extraction

In [ ]:
# Train PCA feature extractor
pca_extractor = PCAFeatureExtractor(n_components=16)

# Flatten images for PCA
flattened_images = sample_images.cpu().numpy().reshape(sample_images.shape[0], -1)

# Train on all training data
all_train_images_flat = train_dataset.data.numpy().reshape(-1, 28*28)
pca_extractor.fit(all_train_images_flat)

print("✓ PCA feature extractor trained")
print(f"  Cumulative explained variance: {pca_extractor.get_explained_variance_ratio()}")

In [ ]:
# Extract features using PCA
pca_features = pca_extractor.transform(all_train_images_flat)

print(f"PCA Features statistics:")
print(f"  Shape: {pca_features.shape}")
print(f"  Min: {pca_features.min():.4f}")
print(f"  Max: {pca_features.max():.4f}")
print(f"  Mean: {pca_features.mean():.4f}")
print(f"  Std: {pca_features.std():.4f}")

In [ ]:
# Compare feature extraction methods
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CNN features
pca_cnn = PCA(n_components=2)
cnn_2d = pca_cnn.fit_transform(all_features)

axes[0].scatter(cnn_2d[:, 0], cnn_2d[:, 1], c=all_labels, cmap='viridis', alpha=0.6, s=50)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].set_title('CNN-based Feature Extraction')
axes[0].grid(True, alpha=0.3)

# PCA features
pca_pca = PCA(n_components=2)
pca_2d = pca_pca.fit_transform(pca_features)

axes[1].scatter(pca_2d[:, 0], pca_2d[:, 1], c=all_labels, cmap='viridis', alpha=0.6, s=50)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('PCA-based Feature Extraction')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Feature Extraction Method Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Quantum Feature Encoding

In [ ]:
# Demonstrate quantum feature encoding
print("Quantum Feature Encoding Pipeline:")
print("="*50)
print(f"1. Original Image: 28×28 = 784 dimensions")
print(f"2. Classical CNN: 784 → 16 features")
print(f"3. Normalization: [0, 1]")
print(f"4. Quantum Encoding: Map to rotation angles")
print(f"5. Quantum Circuit: 4 qubits (each feature → rotation)")
print("="*50)

# Show example encoding
sample_feature = all_features[0]
print(f"\nExample feature vector (first 8):")
print(f"  {sample_feature[:8]}")
print(f"\nQuantum angles (π × feature):")
quantum_angles = sample_feature[:4] * np.pi
print(f"  {quantum_angles}")
print(f"\nRange: [0, π] radians")

## 8. Feature Statistics Summary

In [ ]:
# Class-specific statistics
class_0_features = all_features[all_labels == 0]
class_1_features = all_features[all_labels == 1]

print("Feature Statistics by Class:")
print("="*60)
print(f"Class 0 (digit 0): {len(class_0_features)} samples")
print(f"  Mean: {class_0_features.mean(axis=0)[:4]}... (first 4 features)")
print(f"  Std:  {class_0_features.std(axis=0)[:4]}... (first 4 features)")
print(f"\nClass 1 (digit 1): {len(class_1_features)} samples")
print(f"  Mean: {class_1_features.mean(axis=0)[:4]}... (first 4 features)")
print(f"  Std:  {class_1_features.std(axis=0)[:4]}... (first 4 features)")
print("="*60)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(class_0_features.mean(axis=0), bins=16, alpha=0.6, label='Class 0', edgecolor='black')
axes[0].hist(class_1_features.mean(axis=0), bins=16, alpha=0.6, label='Class 1', edgecolor='black')
axes[0].set_xlabel('Mean Feature Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Mean Features')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

feature_separability = np.abs(class_0_features.mean(axis=0) - class_1_features.mean(axis=0))
axes[1].bar(range(16), feature_separability)
axes[1].set_xlabel('Feature Index')
axes[1].set_ylabel('Mean Difference')
axes[1].set_title('Feature Separability Between Classes')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Ready for Quantum Classification!

In [ ]:
print("\n" + "="*60)
print("✓ PREPROCESSING PIPELINE COMPLETE")
print("="*60)
print(f"\nFeature Matrix Ready for Quantum Processing:")
print(f"  Shape: {all_features.shape}")
print(f"  Data type: {all_features.dtype}")
print(f"  Range: [{all_features.min():.4f}, {all_features.max():.4f}]")
print(f"\nNext Steps:")
print(f"  1. Run: python scripts/train_hybrid.py")
print(f"  2. Train Variational Quantum Classifier with these features")
print(f"  3. Compare with classical baseline")
print("="*60)

## 10. Appendix: Dimensionality Reduction Visualization

In [ ]:
# TSNE visualization (optional, takes longer)
# from sklearn.manifold import TSNE
# print("Computing TSNE visualization (this may take a minute)...")
# tsne = TSNE(n_components=2, random_state=42)
# features_tsne = tsne.fit_transform(all_features)
# plt.figure(figsize=(10, 8))
# plt.scatter(features_tsne[:, 0], features_tsne[:, 1], c=all_labels, cmap='viridis', alpha=0.6)
# plt.title('TSNE Visualization of Extracted Features')
# plt.show()

print("✓ Notebook complete!")